# Jours 3-4-5 - Régression pour la Prédiction des Prix des Vols

## Objectifs
- Construire un modèle de régression performant pour prédire les prix des billets d'avion
- Appliquer des techniques avancées d'ingénierie des caractéristiques
- Optimiser les hyperparamètres des modèles
- Explorer les techniques d'ensemble learning
- Évaluer les performances sur le jeu de test

## Introduction

Dans ce notebook, vous allez développer un modèle de régression pour prédire le prix des billets d'avion en fonction de différentes caractéristiques. Ce projet s'inscrit dans la mission d'AeroAnalytics visant à aider une agence de voyage en ligne à optimiser ses stratégies de tarification.

Vous êtes libre d'explorer différentes approches et techniques pour construire le meilleur modèle possible. Ce notebook vous servira de guide, mais n'hésitez pas à expérimenter et à ajouter vos propres idées !

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import pickle
import sys
import os
import re
from datetime import datetime

# Visualisation avec Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn pour le machine learning
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Modèles de régression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.ensemble import VotingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Ajouter le chemin pour importer utils.py
sys.path.append(os.path.abspath('./'))
from utils import results_predictions_prices

# Pour afficher plus de colonnes dans les DataFrames
pd.set_option('display.max_columns', 30)

## 1. Chargement des données

Si votre ordinateur a des soucis pour faire tourner les modèles de ML, vous pouvez utiliser des versions plus petites des jeux de données. Pour cela, remplacer `train.csv` et `test.csv`, ci-dessous, par:
- `train_50.csv` et `test_50.csv` pour n'utiliser que 50% des données.
- `train_20.csv` et `test_20.csv` pour n'utiliser que 20% des données.

Par contre, utiliser moins de données a tendance à donner de moins bons résultats. Ainsi, lorsque vous avez votre modèle final, n'oubliez pas de remettre toutes les données et de recommencer le notebook.

In [ ]:
# Chargement des données d'entraînement
train_df = pd.read_csv('../../data/flight_prices/train_50.csv')

# Chargement des données de test
test_df = pd.read_csv('../../data/flight_prices/test_50.csv')

# Affichage des premières lignes
train_df.head()

In [ ]:
test_df.head()

In [ ]:
# Vérification des dimensions
print(f"Dimensions du jeu d'entraînement : {train_df.shape}")
print(f"Dimensions du jeu de test : {test_df.shape}")

In [ ]:
# Vérification des valeurs manquantes pour train et test
missing_values_train = train_df.isnull().sum()
missing_values_test = test_df.isnull().sum()
missing_values_pct_train = (missing_values_train / len(train_df)) * 100
missing_values_pct_test = (missing_values_test / len(test_df)) * 100

# Création d'un DataFrame pour visualiser les valeurs manquantes
missing_df = pd.DataFrame({
    'Train - Nombre': missing_values_train,
    'Train - %': missing_values_pct_train,
    'Test - Nombre': missing_values_test, 
    'Test - %': missing_values_pct_test
})

# Affichage des colonnes avec des valeurs manquantes
missing_df[
    (missing_df['Train - Nombre'] > 0) | 
    (missing_df['Test - Nombre'] > 0)
].sort_values('Train - Nombre', ascending=False)

### Visualisation des données

Explorez les données pour mieux comprendre la distribution des variables et leurs relations avec le prix des billets.

In [ ]:
# Distribution de la variable cible (Prix)
fig = px.histogram(train_df, x='Price', nbins=50, title='Distribution des prix des billets d\'avion')
fig.show()

In [ ]:
# Exemple de visualisation : prix moyen par compagnie aérienne
airline_price = train_df.groupby('Airline')['Price'].mean().reset_index().sort_values('Price', ascending=False)

fig = px.bar(airline_price, x='Airline', y='Price', 
             title='Prix moyen par compagnie aérienne',
             color='Price',
             color_continuous_scale=px.colors.sequential.Viridis)
fig.show()

In [ ]:
# Exemple de visualisation : prix moyen par nombre d'escales
stops_price = train_df.groupby('Number Of Stops')['Price'].mean().reset_index()

fig = px.bar(stops_price, x='Number Of Stops', y='Price', 
             title='Prix moyen par nombre d\'escales',
             color='Price',
             color_continuous_scale=px.colors.sequential.Viridis)
fig.show()

## 2. Ingénierie des caractéristiques

L'ingénierie des caractéristiques est une étape cruciale pour améliorer les performances des modèles de régression. Dans cette section, nous allons explorer différentes techniques pour créer de nouvelles caractéristiques pertinentes à partir des données existantes.

### 2.1 Traitement des dates

Commençons par extraire des informations utiles à partir des colonnes de dates.

In [ ]:
# Copie des DataFrames pour ne pas modifier les originaux
train_processed = train_df.copy()
test_processed = test_df.copy()

# Conversion des colonnes de dates en format datetime
date_columns = ['Searched Date', 'Departure Date', 'Arrival Date']

for col in date_columns:
    train_processed[col] = pd.to_datetime(train_processed[col])
    test_processed[col] = pd.to_datetime(test_processed[col])

# Extraction de caractéristiques à partir des dates de départ
# Jour de la semaine
train_processed['Departure Day of Week'] = train_processed['Departure Date'].dt.dayofweek
test_processed['Departure Day of Week'] = test_processed['Departure Date'].dt.dayofweek

# Mois
train_processed['Departure Month'] = train_processed['Departure Date'].dt.month
test_processed['Departure Month'] = test_processed['Departure Date'].dt.month

# Heure de la journée
train_processed['Departure Hour'] = train_processed['Departure Date'].dt.hour
test_processed['Departure Hour'] = test_processed['Departure Date'].dt.hour

# Jours entre la recherche et le départ
train_processed['Days Before Departure'] = (train_processed['Departure Date'] - train_processed['Searched Date']).dt.days
test_processed['Days Before Departure'] = (test_processed['Departure Date'] - test_processed['Searched Date']).dt.days

# Affichage des nouvelles caractéristiques
train_processed[['Departure Day of Week', 'Departure Month', 'Departure Hour', 'Days Before Departure']].head()

### 2.2 Traitement de la durée du vol

In [ ]:
# Calcul de la durée du vol en heures
train_processed['Flight Duration (hours)'] = (train_processed['Arrival Date'] - train_processed['Departure Date']).dt.total_seconds() / 3600
test_processed['Flight Duration (hours)'] = (test_processed['Arrival Date'] - test_processed['Departure Date']).dt.total_seconds() / 3600

# Correction pour les vols qui atterrissent le lendemain
# Note: Cette correction n'est pas nécessaire si 'Flight Lands Next Day' est déjà pris en compte dans les dates

# Affichage de la distribution de la durée des vols
fig = px.histogram(train_processed, x='Flight Duration (hours)', nbins=50, 
                   title='Distribution de la durée des vols (en heures)')
fig.show()

### 2.3 Traitement des variables catégorielles

In [11]:
# Mapping dictionary
cabin_mapping = {
    'Basic Economy': 'Basic Economy',
    'Blue Basic': 'Basic Economy',
    'UltraBasic': 'Basic Economy',
    'Main Cabin Basic': 'Basic Economy',
    'PorterClassic Basic': 'Basic Economy',
    'Discount': 'Basic Economy',

    'Economy': 'Economy',
    'Saver': 'Economy',
    'Main Cabin': 'Economy',
    'Main': 'Economy',
    'Standard': 'Economy',
    'Blue': 'Economy',
    'Mixed': 'Economy',

    'Economy Plus': 'Premium Economy',
    'Comfort +': 'Premium Economy',
    'Premium Economy': 'Premium Economy',

    'Business/First': 'Business',
    'BizFare': 'Business',
    'Mint': 'Business',

    'First': 'First Class',
    'Business/First (fully refundable)': 'First Class',

    'Blue Refundable': 'Refundable Options',
    'Economy (fully refundable)': 'Refundable Options',
    'Main Cabin Flex': 'Refundable Options',
    'Main Select': 'Refundable Options',
    'No Flex Fare': 'Refundable Options'
}

# Map the original cabin categories to the new categories
train_processed['Cabin Category'] = train_processed['Cabin'].map(cabin_mapping)
test_processed['Cabin Category'] = test_processed['Cabin'].map(cabin_mapping)

In [12]:
# Il y a peut être quelque chose à faire avec les routes. =)

### 2.4 Encodage des variables catégorielles

In [ ]:
# Identification des variables catégorielles
categorical_features = ['Airline', 'Cabin Category']

# Récupération de toutes les valeurs uniques des ensembles d'entraînement et de test
all_categories = {}
for feature in categorical_features:
    train_categories = set(train_processed[feature].unique())
    test_categories = set(test_processed[feature].unique())
    all_categories[feature] = train_categories.union(test_categories)

# Création de DataFrames vides avec toutes les catégories possibles
train_encoded = train_processed.copy()
test_encoded = test_processed.copy()

# Encodage de chaque variable catégorielle
for feature in categorical_features:
    # Création des variables indicatrices pour toutes les catégories possibles
    train_dummies = pd.get_dummies(train_processed[feature], prefix=feature, drop_first=True)
    test_dummies = pd.get_dummies(test_processed[feature], prefix=feature, drop_first=True)
    
    # Ajout des colonnes manquantes dans chaque ensemble
    missing_in_train = set(test_dummies.columns) - set(train_dummies.columns)
    missing_in_test = set(train_dummies.columns) - set(test_dummies.columns)
    
    for col in missing_in_train:
        train_dummies[col] = 0
    for col in missing_in_test:
        test_dummies[col] = 0
        
    # Assurance du même ordre des colonnes
    train_dummies = train_dummies.reindex(columns=sorted(train_dummies.columns))
    test_dummies = test_dummies.reindex(columns=sorted(test_dummies.columns))
    
    # Suppression de la colonne d'origine et ajout des colonnes encodées
    train_encoded = train_encoded.drop(feature, axis=1)
    test_encoded = test_encoded.drop(feature, axis=1)
    train_encoded = pd.concat([train_encoded, train_dummies], axis=1)
    test_encoded = pd.concat([test_encoded, test_dummies], axis=1)

# Vérification des colonnes après encodage
print(f"Nombre de colonnes après encodage : {train_encoded.shape[1]}")
print("Nouvelles colonnes créées :")
new_columns = [col for col in train_encoded.columns if col not in train_processed.columns]
print(new_columns[:10])  # Affichage des 10 premières nouvelles colonnes

### 2.5 Normalisation des variables numériques

In [ ]:
# Identification des variables numériques
numerical_features = ['Departure Month', 'Departure Hour', 'Days Before Departure', 
                      'Flight Duration (hours)', 'Number Of Stops', 'Departure Day of Week']

# Création d'un scaler
scaler = StandardScaler()

# Application du scaler aux caractéristiques numériques
train_encoded[numerical_features] = scaler.fit_transform(train_encoded[numerical_features])
test_encoded[numerical_features] = scaler.transform(test_encoded[numerical_features])

# Vérification des résultats
train_encoded[numerical_features].describe()

### 2.6 Préparation des données pour l'entraînement

In [ ]:
# Suppression des colonnes de dates qui ne sont plus nécessaires
columns_to_drop = ['Searched Date', 'Departure Date', 'Arrival Date', 'Departure Airport', 'Arrival Airport', 'Flight Lands Next Day', 'Route', 'Cabin']

# Séparation des caractéristiques et de la variable cible
X_train = train_encoded.drop(['Price'] + columns_to_drop, axis=1)
y_train = train_encoded['Price']

# Préparation des données de test
X_test = test_encoded.drop(columns_to_drop, axis=1)

# Vérification des dimensions
print(f"Dimensions de X_train : {X_train.shape}")
print(f"Dimensions de y_train : {y_train.shape}")
print(f"Dimensions de X_test : {X_test.shape}")

In [ ]:
X_test

In [17]:
# A cause du nom des Aéroports, il y a des caractères spéciaux qui posent problème. On les remplace par des underscores.

# Function to check for special JSON characters
def find_special_json_chars(column_name):
    # Characters that might cause issues in JSON
    special_chars = r'[\{\}\[\]"\\:,]'
    return bool(re.search(special_chars, column_name))

to_be_renamed = {}

for col in X_train.columns:
    if find_special_json_chars(col):
        to_be_renamed[col] = re.sub(r'[\{\}\[\]"\\:,]', '_', col)

X_train = X_train.rename(columns=to_be_renamed)
X_test = X_test.rename(columns=to_be_renamed)

## 3. Modélisation et évaluation

Dans cette section, nous allons entraîner différents modèles de régression et évaluer leurs performances.

### 3.1 Validation croisée

Commençons par diviser nos données d'entraînement en ensembles d'entraînement et de validation pour évaluer nos modèles.

In [ ]:
# Division des données d'entraînement en ensembles d'entraînement et de validation
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Dimensions de X_train_split : {X_train_split.shape}")
print(f"Dimensions de X_val : {X_val.shape}")
print(f"Dimensions de y_train_split : {y_train_split.shape}")
print(f"Dimensions de y_val : {y_val.shape}")

### 3.2 Entraînement de modèles de base

Entraînons plusieurs modèles de régression et comparons leurs performances.

In [ ]:
# Définition des modèles à tester
models = {
    'Régression Linéaire': LinearRegression(),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=50, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=50, random_state=42)
}

# Dictionnaire pour stocker les résultats
results = {}

# Entraînement et évaluation de chaque modèle
for name, model in models.items():
    print(f"Entraînement du modèle : {name}")
    
    # Entraînement du modèle
    model.fit(X_train_split, y_train_split)
    
    # Prédictions sur l'ensemble de validation
    y_pred = model.predict(X_val)
    
    # Calcul des métriques
    mae_val = mean_absolute_error(y_val, y_pred)
    mse_val = mean_squared_error(y_val, y_pred)
    rmse_val = np.sqrt(mse_val)
    r2_val = r2_score(y_val, y_pred)
    
    # Stockage des résultats
    results[name] = {
        'MAE': mae_val,
        'MSE': mse_val,
        'RMSE': rmse_val,
        'R²': r2_val
    }
    
    print(f"MAE : {mae_val:.2f}")
    print(f"MSE : {mse_val:.2f}")
    print(f"RMSE : {rmse_val:.2f}")
    print(f"R² : {r2_val:.4f}")
    print("\n" + "-"*50 + "\n")

In [ ]:
# Visualisation des résultats
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Modèle', 'MAE', 'MSE', 'RMSE', 'R²']

# Tri des modèles par MAE croissant (meilleur en premier)
results_df = results_df.sort_values('MAE')

# Création d'un graphique pour comparer les MAE
fig = px.bar(results_df, x='Modèle', y='MAE', 
             title='Comparaison des MAE des différents modèles',
             color='MAE',
             color_continuous_scale=px.colors.sequential.Viridis_r)
fig.show()

In [ ]:
# Visualisation des prédictions vs valeurs réelles pour le meilleur modèle
# Supposons que le meilleur modèle soit Random Forest (à adapter selon vos résultats)
best_model_name = results_df.iloc[0]['Modèle']  # Le modèle avec le MAE le plus bas
best_model = models[best_model_name]

# Prédictions sur l'ensemble de validation
y_pred_best = best_model.predict(X_val)

# Création d'un DataFrame pour la visualisation
pred_df = pd.DataFrame({
    'Valeur réelle': y_val,
    'Prédiction': y_pred_best
})

# Graphique de dispersion
fig = px.scatter(pred_df, x='Valeur réelle', y='Prédiction',
                title=f'Prédictions vs Valeurs réelles - {best_model_name}',
                opacity=0.6)

# Ajout d'une ligne de référence y=x
fig.add_trace(
    go.Scatter(x=[pred_df['Valeur réelle'].min(), pred_df['Valeur réelle'].max()],
              y=[pred_df['Valeur réelle'].min(), pred_df['Valeur réelle'].max()],
              mode='lines', name='y=x', line=dict(color='red', dash='dash'))
)

fig.show()

### 3.3 Optimisation des hyperparamètres (Optionnel)

Optimisons les hyperparamètres du meilleur modèle pour améliorer ses performances.

In [22]:
# Exemple d'optimisation des hyperparamètres pour Random Forest
# Définition de la grille de paramètres à tester
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Création du modèle Random Forest
rf = RandomForestRegressor(random_state=42)

# Création de la recherche par grille avec validation croisée
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                          cv=3, n_jobs=-1, verbose=1, scoring='neg_mean_absolute_error')

# Entraînement du modèle avec recherche par grille
# Attention : cette opération peut prendre du temps !
# grid_search.fit(X_train, y_train)

# Affichage des meilleurs paramètres
# print(f"Meilleurs paramètres : {grid_search.best_params_}")
# print(f"Meilleur MAE : {-grid_search.best_score_:.2f}")

# Utilisation du meilleur modèle
# best_rf = grid_search.best_estimator_

### 3.4 Ensemble Learning (Optionnel)

Explorons les techniques d'ensemble learning pour améliorer davantage les performances.

Attention, cela prend du temps à calculer !

In [23]:
# Sélection des meilleurs modèles
estimators = [
    ('rf', RandomForestRegressor(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=50, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=50, random_state=42))
]

In [ ]:
# Exemple de Voting Regressor (Moyenne entre les modèles)

# Création du Voting Regressor (moyenne entre les modèles)
# On peut ajouter des poids à chaque modèle pour donner plus ou moins d'importance à chaque modèle.
voting_reg = VotingRegressor(estimators=estimators, weights=(0.6, 0.2, 0.2), n_jobs=-1,verbose=True)

# Entraînement du modèle
voting_reg.fit(X_train_split, y_train_split)

# Prédictions sur l'ensemble de validation
y_pred_voting = voting_reg.predict(X_val)

# Calcul des métriques
mae_voting = mean_absolute_error(y_val, y_pred_voting)
mse_voting = mean_squared_error(y_val, y_pred_voting)
rmse_voting = np.sqrt(mse_voting)
r2_voting = r2_score(y_val, y_pred_voting)

print(f"MAE du Voting Regressor : {mae_voting:.2f}")
print(f"MSE du Voting Regressor : {mse_voting:.2f}")
print(f"RMSE du Voting Regressor : {rmse_voting:.2f}")
print(f"R² du Voting Regressor : {r2_voting:.4f}")

### 3.5 Évaluation sur le jeu de test

Utilisons maintenant notre meilleur modèle pour faire des prédictions sur le jeu de test.

In [ ]:
# Sélection du meilleur modèle (à adapter selon vos résultats)
best_model = voting_reg  # ou un autre modèle de votre choix

# Entraînement du modèle sur l'ensemble des données d'entraînement
best_model.fit(X_train, y_train)

# Prédictions sur le jeu de test
y_pred_test = best_model.predict(X_test)

# Affichage des premières prédictions
pd.DataFrame({'Prix prédit': y_pred_test}).head(10)

In [ ]:
# Utilisation de la fonction results_predictions_prices pour évaluer les performances
# Cette fonction va charger les vrais prix du jeu de test et calculer le MAE
mae_test = results_predictions_prices(y_pred_test)
print(f"MAE sur le jeu de test : {mae_test:.5f}")